# DAPT Contriever: Continued Pre-Training on PubMed Abstracts

Continues Contriever's contrastive pre-training on PubMed abstracts using the original random cropping objective (64-token spans). Trains on Colab T4 GPU.

In [1]:
# Install dependencies
#!pip install torch==2.1.0 --index-url https://download.pytorch.org/whl/cu118 --upgrade -q
!pip install datasets transformers faiss-cpu rank_bm25 numpy tqdm psutil -q
print("Done")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 56.0 MB/s eta 0:00:00
Done


In [3]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from tqdm import tqdm
import faiss
import random
import json
import time
import gc
import os

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

CUDA available: True
GPU: Tesla T4


## 1. Load Training Corpus

Use PubMedQA artificial split (~211K abstracts) as the domain-specific pre-training corpus.

In [4]:
# Load pre-training corpus from PubMedQA artificial split
print("Loading PubMed abstracts for pre-training...")
dapt_corpus_raw = load_dataset("qiaojin/PubMedQA", "pqa_artificial", split="train")
print(f"Loaded {len(dapt_corpus_raw)} abstracts")

# Extract abstracts
dapt_texts = []
for example in tqdm(dapt_corpus_raw, desc="Extracting abstracts"):
    abstract_sections = example["context"]["contexts"]
    abstract = " ".join(abstract_sections)
    if len(abstract.strip()) > 50:  # filter very short ones
        dapt_texts.append(abstract)

print(f"Training corpus: {len(dapt_texts)} abstracts")
print(f"Example: {dapt_texts[0][:200]}...")

Loading PubMed abstracts for pre-training...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

pqa_artificial/train-00000-of-00001.parq(…):   0%|          | 0.00/233M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/211269 [00:00<?, ? examples/s]

Loaded 211269 abstracts


Extracting abstracts: 100%|██████████| 211269/211269 [00:29<00:00, 7278.07it/s]

Training corpus: 211266 abstracts
Example: Chronic rhinosinusitis (CRS) is a heterogeneous disease with an uncertain pathogenesis. Group 2 innate lymphoid cells (ILC2s) represent a recently discovered cell population which has been implicated ...


## 2. Contrastive Dataset with Random Cropping

Implements Contriever's original training objective:
- For each document, randomly crop two spans of **64 tokens**
- These form a positive pair (anchor, positive)
- In-batch negatives: other passages in the same batch serve as negatives
- Loss: in-batch contrastive (NT-Xent / InfoNCE)

In [5]:
class RandomCropDataset(Dataset):
    """
    Implements Contriever's random cropping objective.
    For each abstract, creates two random token-level crops as a positive pair.
    Span size: 64 tokens (default from Contriever paper).
    """
    def __init__(self, texts, tokenizer, span_tokens=64, max_length=512):
        self.texts = texts
        self.tokenizer = tokenizer
        self.span_tokens = span_tokens
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]

        # Tokenize the full abstract
        tokens = self.tokenizer.encode(text, add_special_tokens=False, max_length=self.max_length, truncation=True)

        # If too short to crop, just use the full text twice
        if len(tokens) <= self.span_tokens:
            span1 = tokens
            span2 = tokens
        else:
            # Random crop 1
            start1 = random.randint(0, len(tokens) - self.span_tokens)
            span1 = tokens[start1:start1 + self.span_tokens]

            # Random crop 2 (independent)
            start2 = random.randint(0, len(tokens) - self.span_tokens)
            span2 = tokens[start2:start2 + self.span_tokens]

        return {
            "span1": span1,
            "span2": span2
        }

def collate_fn(batch):
    """Pad spans to same length within batch."""
    tokenizer = collate_fn.tokenizer

    spans1 = [item["span1"] for item in batch]
    spans2 = [item["span2"] for item in batch]

    def pad_batch(spans):
        max_len = max(len(s) for s in spans)
        input_ids = []
        attention_masks = []
        for s in spans:
            pad_len = max_len - len(s)
            input_ids.append(s + [tokenizer.pad_token_id] * pad_len)
            attention_masks.append([1] * len(s) + [0] * pad_len)
        return (
            torch.tensor(input_ids, dtype=torch.long),
            torch.tensor(attention_masks, dtype=torch.long)
        )

    ids1, masks1 = pad_batch(spans1)
    ids2, masks2 = pad_batch(spans2)

    return {
        "input_ids1": ids1, "attention_mask1": masks1,
        "input_ids2": ids2, "attention_mask2": masks2,
    }

print("Dataset class defined.")

Dataset class defined.


## 3. Mean Pooling + Contrastive Loss

In [6]:
def mean_pool(token_embeddings, attention_mask):
    """Mean pooling over token embeddings (Contriever's method)."""
    mask = attention_mask.unsqueeze(-1).float()
    return (token_embeddings * mask).sum(dim=1) / mask.sum(dim=1)

def contrastive_loss(emb1, emb2, temperature=0.05):
    """
    In-batch NT-Xent contrastive loss.
    emb1, emb2: [batch_size, hidden_dim] — positive pairs
    Negatives: all other pairs in the batch
    Temperature: 0.05 (from Contriever paper)
    """
    # Normalize
    emb1 = torch.nn.functional.normalize(emb1, dim=-1)
    emb2 = torch.nn.functional.normalize(emb2, dim=-1)

    # Similarity matrix [batch_size, batch_size]
    sim = torch.matmul(emb1, emb2.T) / temperature

    # Labels: diagonal is positive pair
    labels = torch.arange(sim.size(0), device=sim.device)

    # Symmetric loss
    loss = (torch.nn.functional.cross_entropy(sim, labels) +
            torch.nn.functional.cross_entropy(sim.T, labels)) / 2

    return loss

print("Loss function defined.")

Loss function defined.


## 4. Training Setup

In [7]:
# Hyperparameters
BATCH_SIZE = 64        # fits on T4 16GB
LEARNING_RATE = 1e-5
NUM_EPOCHS = 3
SPAN_TOKENS = 64       # from Contriever paper
TEMPERATURE = 0.05     # from Contriever paper
MAX_STEPS_PER_EPOCH = 500  # ~32K examples per epoch, cap for speed

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load model
model_name = "facebook/contriever"
print(f"Loading {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
model.to(device)
model.train()
print(f"Model loaded on {device}")

# Attach tokenizer to collate function
collate_fn.tokenizer = tokenizer

# Dataset and dataloader
dataset = RandomCropDataset(dapt_texts, tokenizer, span_tokens=SPAN_TOKENS)
dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
print(f"Dataset size: {len(dataset)} abstracts")
print(f"Steps per epoch (capped): {MAX_STEPS_PER_EPOCH}")

Loading facebook/contriever...


config.json:   0%|          | 0.00/619 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/321 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: facebook/contriever
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Model loaded on cuda
Dataset size: 211266 abstracts
Steps per epoch (capped): 500


## 5. Training Loop

In [8]:
os.makedirs("checkpoints", exist_ok=True)

training_log = []
best_loss = float("inf")

for epoch in range(NUM_EPOCHS):
    model.train()
    epoch_loss = 0.0
    steps = 0
    start_time = time.time()

    for batch in tqdm(dataloader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}", total=MAX_STEPS_PER_EPOCH):
        if steps >= MAX_STEPS_PER_EPOCH:
            break

        # Move to device
        ids1 = batch["input_ids1"].to(device)
        masks1 = batch["attention_mask1"].to(device)
        ids2 = batch["input_ids2"].to(device)
        masks2 = batch["attention_mask2"].to(device)

        # Encode both spans
        out1 = model(input_ids=ids1, attention_mask=masks1)
        out2 = model(input_ids=ids2, attention_mask=masks2)

        emb1 = mean_pool(out1.last_hidden_state, masks1)
        emb2 = mean_pool(out2.last_hidden_state, masks2)

        loss = contrastive_loss(emb1, emb2, temperature=TEMPERATURE)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        epoch_loss += loss.item()
        steps += 1

        # Free memory
        del ids1, masks1, ids2, masks2, out1, out2, emb1, emb2, loss

    avg_loss = epoch_loss / steps
    elapsed = time.time() - start_time
    print(f"Epoch {epoch+1}: avg_loss={avg_loss:.4f}, steps={steps}, time={elapsed:.0f}s")

    training_log.append({"epoch": epoch+1, "loss": avg_loss, "steps": steps})

    # Save checkpoint
    checkpoint_path = f"checkpoints/contriever_dapt_epoch{epoch+1}"
    model.save_pretrained(checkpoint_path)
    tokenizer.save_pretrained(checkpoint_path)
    print(f"Saved checkpoint: {checkpoint_path}")

    if avg_loss < best_loss:
        best_loss = avg_loss
        model.save_pretrained("checkpoints/contriever_dapt_best")
        tokenizer.save_pretrained("checkpoints/contriever_dapt_best")
        print(f"New best model saved (loss={best_loss:.4f})")

with open("dapt_training_log.json", "w") as f:
    json.dump(training_log, f, indent=2)
print("\nTraining complete. Log saved to dapt_training_log.json")

Epoch 1/3: 100%|██████████| 500/500 [12:11<00:00,  1.46s/it]

Epoch 1: avg_loss=0.4218, steps=500, time=732s


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved checkpoint: checkpoints/contriever_dapt_epoch1


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Epoch 3: avg_loss=0.2596, steps=500, time=752s


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved checkpoint: checkpoints/contriever_dapt_epoch3


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

New best model saved (loss=0.2596)

Training complete. Log saved to dapt_training_log.json


## 6. Evaluate DAPT Contriever on PubMedQA

Load the best checkpoint and evaluate retrieval on the expanded corpus (1K gold + 10K distractors).

In [9]:
def encode_texts(texts, tokenizer, model, batch_size=64, desc="Encoding"):
    """Encode with mean pooling."""
    model.eval()
    all_embeddings = []
    for i in tqdm(range(0, len(texts), batch_size), desc=desc):
        batch = texts[i:i + batch_size]
        inputs = tokenizer(batch, padding=True, truncation=True, max_length=512, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs)
        emb = mean_pool(outputs.last_hidden_state, inputs["attention_mask"])
        all_embeddings.append(emb.cpu().numpy())
        del inputs, outputs, emb
    return np.vstack(all_embeddings)

def evaluate_retrieval(retrieved_ids, gold_ids, k_values=[5, 20]):
    recalls = {k: 0.0 for k in k_values}
    mrr_sum = 0.0
    n = len(gold_ids)
    for i in range(n):
        gold = gold_ids[i]
        retrieved = retrieved_ids[i]
        for rank, doc_id in enumerate(retrieved):
            if doc_id == gold:
                mrr_sum += 1.0 / (rank + 1)
                break
        for k in k_values:
            if gold in retrieved[:k]:
                recalls[k] += 1.0
    results = {f"Recall@{k}": recalls[k] / n for k in k_values}
    results["MRR"] = mrr_sum / n
    return results

In [10]:
# Load PubMedQA labeled + distractors
print("Loading evaluation data...")
labeled = load_dataset("qiaojin/PubMedQA", "pqa_labeled", split="train")
distractors_raw = load_dataset("qiaojin/PubMedQA", "pqa_artificial", split="train")

NUM_DISTRACTORS = 10000

queries = [ex["question"] for ex in labeled]
gold_abstracts = [" ".join(ex["context"]["contexts"]) for ex in labeled]
gold_passage_ids = list(range(len(gold_abstracts)))

corpus = list(gold_abstracts)
for i, ex in enumerate(distractors_raw):
    if i >= NUM_DISTRACTORS:
        break
    corpus.append(" ".join(ex["context"]["contexts"]))

print(f"Queries: {len(queries)}, Corpus: {len(corpus)}")

Loading evaluation data...


pqa_labeled/train-00000-of-00001.parquet:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Queries: 1000, Corpus: 11000


In [11]:
# Load best DAPT checkpoint
print("Loading best DAPT checkpoint...")
dapt_tokenizer = AutoTokenizer.from_pretrained("checkpoints/contriever_dapt_best")
dapt_model = AutoModel.from_pretrained("checkpoints/contriever_dapt_best")
dapt_model.to(device)

# Encode
print("Encoding corpus...")
corpus_emb = encode_texts(corpus, dapt_tokenizer, dapt_model, desc="Corpus")

print("Encoding queries...")
query_emb = encode_texts(queries, dapt_tokenizer, dapt_model, desc="Queries")

# Free model
del dapt_model
gc.collect()
torch.cuda.empty_cache()

# FAISS search
faiss.normalize_L2(corpus_emb)
faiss.normalize_L2(query_emb)
index = faiss.IndexFlatIP(corpus_emb.shape[1])
index.add(corpus_emb)
_, retrieved = index.search(query_emb, 20)
dapt_retrieved = [r.tolist() for r in retrieved]

dapt_results = evaluate_retrieval(dapt_retrieved, gold_passage_ids)
print("\nDAPT Contriever Results:")
for metric, value in dapt_results.items():
    print(f"  {metric}: {value:.4f}")

# Save
np.save("corpus_emb_dapt.npy", corpus_emb)
np.save("query_emb_dapt.npy", query_emb)
json.dump(dapt_results, open("dapt_results.json", "w"), indent=2)

Loading best DAPT checkpoint...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Encoding corpus...


Corpus: 100%|██████████| 172/172 [06:42<00:00,  2.34s/it]


Encoding queries...


Queries: 100%|██████████| 16/16 [00:02<00:00,  5.50it/s]



DAPT Contriever Results:
  Recall@5: 0.9490
  Recall@20: 0.9820
  MRR: 0.8925


## 7. Save to Google Drive

In [13]:
# Mount Google Drive and save checkpoints + results
from google.colab import drive
drive.mount('/content/drive')

import shutil
save_dir = "/content/drive/MyDrive/cs505_project"
os.makedirs(save_dir, exist_ok=True)

# Save best checkpoint
shutil.copytree("checkpoints/contriever_dapt_best", f"{save_dir}/contriever_dapt_best", dirs_exist_ok=True)

# Save results and embeddings
shutil.copy("dapt_results.json", save_dir)
shutil.copy("dapt_training_log.json", save_dir)
np.save(f"{save_dir}/corpus_emb_dapt.npy", corpus_emb)
np.save(f"{save_dir}/query_emb_dapt.npy", query_emb)

print(f"All files saved to {save_dir}")

Mounted at /content/drive
All files saved to /content/drive/MyDrive/cs505_project
